# Solving finite discrete-action games in Stata with `dagamesolve`

### Theory, numerical methods, and examples

**Matthew J. Baker**  
Hunter College and the Graduate Center, CUNY

*Draft: October 2026*

---

## Abstract

`dagamesolve` is a Stata/Mata package for solving finite, simultaneous-move,
discrete-action games of complete information. The package supports an arbitrary
number of players, allows players to have different numbers of actions, searches
for pure, partially mixed, and fully mixed Nash equilibria, and can solve a
sequence of games with a common action structure in a single call.

The computational difficulty is not usually the detection of pure-strategy
equilibria. The hard part is the systematic search for mixed-strategy equilibria.
On a particular support, equilibrium requires solving a system of nonlinear
polynomial equations, and a complete search must also consider the relevant
supports. `dagamesolve` combines dominance reduction, direct checks for pure
equilibria, support enumeration, interval methods for the nonlinear systems,
and Newton refinement. A faster point-based Newton search is also available.

This document explains the representation of games used by the package,
describes the numerical ideas underlying the solver, and provides executable
Stata examples. In particular, it emphasizes a feature that is useful for
econometric applications: a sequence of related games can be constructed and
solved directly in a Stata dataset.


> **About this notebook.**  
> This is intended to be both a technical note and an executable document.
> Markdown cells contain the exposition and equations; Stata cells contain
> examples that can be run directly with a Stata Jupyter kernel. The notebook
> is the master source. A PDF can later be generated from the executed notebook.
>
> The theoretical discussion is adapted from the earlier presentation
> *Finding all solutions to discrete games (using Stata)* and updated to match
> the current `dagamesolve` and `dagamestrats` interfaces.


## 1. Setup

The examples assume that `dagamesolve`, `dagamestrats`, and their dependencies
are already installed or that the notebook is being run from a location where
the development version of the package is on Stata's ado-path.

The package currently depends on `moremata`, `int_utils`, `rowmat_utils`, and
`intsolver`. The `dependency.do` file in the `DaGameSolve` repository installs
the required dependencies.

The following cell simply checks the Stata environment and locates the two main
commands.


In [ ]:
clear all
set more off

about
which dagamesolve
which dagamestrats

## 2. Why solve finite games inside Stata?

There are at least three motivations.

First, finding **all** Nash equilibria of a finite game can itself be a
nontrivial computational problem. Pure-strategy profiles can be checked
directly, but mixed strategies require solving nonlinear systems and considering
alternative supports.

Second, many empirical models require a **sequence of games**, not a single
isolated game. Entry models are a natural example. Different markets may contain
different payoff shifters, but the strategic structure is common. If the game
solver is embedded in Stata, game construction, equilibrium computation, data
analysis, and eventually estimation can live in the same environment.

Third, sequences of games are useful even outside estimation. They make it easy
to study how the number and nature of equilibria change as payoff parameters,
the number of players, or the number of available actions change.

This was one of the original motivations for `dagamesolve`: not merely to solve
one textbook game, but to make equilibrium computation usable as a component
inside a larger empirical workflow.


## 3. Finite discrete-action games

Consider a complete-information simultaneous-move game with players $i \in N$ players where each player has a set of discrete actions:

$$
A_i=\{1,\ldots,K_i\}.
$$

A mixed strategy for player $i$ is a probability vector

$$
\sigma_i=(\sigma_{i1},\ldots,\sigma_{iK_i}),
\qquad
\sigma_{ik}\geq 0,
\qquad
\sum_{k=1}^{K_i}\sigma_{ik}=1.
$$

A strategy profile $$\sigma^*=(\sigma_1^*,\ldots,\sigma_N^*)$$ is a Nash
equilibrium if no player can increase expected payoff through a unilateral
deviation:

$$
u_i(\sigma_i^*,\sigma_{-i}^*)
\geq
u_i(\sigma_i',\sigma_{-i}^*)
\quad
\text{for every feasible }\sigma_i'.
$$

Pure strategies are special cases in which one action receives a weight of probability equal to unity.
A partially mixed equilibrium has some players mixing and others playing pure
strategies. A fully mixed equilibrium places positive probability on every
action in the relevant support.

Mixed strategies matter computationally as well as theoretically. Nash's
existence theorem permits mixed strategies, and equilibrium search cannot in
general stop after inspecting the finite collection of pure profiles.


### 3.1 A two-action illustration of existence

For intuition, suppose each player has two actions and write $$\sigma_i$$ for
the probability of choosing action 1. One way of motivating the fixed-point
argument is to define the gain from shifting toward action 1,

$$
g_i(\sigma)
=
\max\{u_i(1,\sigma_{-i})-u_i(2,\sigma_{-i}),0\},
$$

and construct a continuous adjustment mapping on the mixed-strategy simplex.
A fixed point of an appropriate mapping has the property that profitable
unilateral adjustments have been exhausted.

The important computational lesson is not the particular fixed-point proof.
It is that equilibrium may lie in the interior of a probability simplex rather
than at one of its vertices. A numerical solver therefore has to deal with
continuous probability variables even though the underlying actions are
discrete.


## 4. Representing a game in "list" form

Games are often displayed in normal form. For computation, `dagamesolve` uses a
list representation.

Let $A$ denote a matrix whose columns enumerate the pure-strategy action
profiles and let $\Pi$ contain the corresponding payoffs. If there are $N$
players, both matrices have $N$ rows. The number of columns is

$$
M=\prod_{i=1}^{N} K_i.
$$

For a two-player, two-action game, the action-profile matrix is

$$
A=
\begin{bmatrix}
1&1&2&2\\
1&2&1&2
\end{bmatrix}.
$$

Each column is one complete pure-strategy profile. The payoff matrix has exactly
the same column ordering.

In a Stata dataset, players are observations and action profiles are variables.
The utility command `dagamestrats` constructs the action-profile variables
automatically.


### Example: constructing action profiles

Suppose player 1 has two actions, player 2 has three, and player 3 has two.
There are \(2\times3\times2=12\) pure-strategy profiles.


In [ ]:
clear
set obs 3

gen game = 1
gen acts = 2
replace acts = 3 in 2

dagamestrats acts, group(game) generate(a)

list game acts a*
return list


The generated variables `a1`, `a2`, ..., `a12` are the twelve pure-strategy
profiles. Corresponding payoff variables can be created in the same order and
passed to `dagamesolve`.

The same organization extends naturally to a **sequence of games**. Each game
occupies a group of observations, one observation per player. Games in a single
call must have the same number of players and the same action structure, with
corresponding players ordered consistently across games.


## 5. Why mixed-strategy equilibrium is the difficult part

Suppose, for simplicity, that three players each have two actions. Let
\(p_i\) be the probability that player \(i\) chooses action 1.

If all three players mix, each player must be indifferent between the two
actions. For player 1 this condition has the form

\[
\begin{aligned}
& p_2p_3\,u_1(1,1,1)
+p_2(1-p_3)\,u_1(1,1,2) \\
&\quad +(1-p_2)p_3\,u_1(1,2,1)
+(1-p_2)(1-p_3)\,u_1(1,2,2) \\
={}&
p_2p_3\,u_1(2,1,1)
+p_2(1-p_3)\,u_1(2,1,2) \\
&\quad +(1-p_2)p_3\,u_1(2,2,1)
+(1-p_2)(1-p_3)\,u_1(2,2,2).
\end{aligned}
\]

There is one analogous indifference equation for each mixing player. Even in a
binary-action game these equations are polynomial in the other players'
probabilities.

That is only part of the problem. A game may have equilibria on smaller
supports: one player may mix while another plays a pure strategy, or different
players may mix over different subsets of their actions. A systematic search
therefore has to consider candidate supports as well as solve the nonlinear
systems associated with them.

The number of pure profiles and the number of possible supports both grow
rapidly as players and actions are added.


## 6. The solution strategy used by `dagamesolve`

At a high level, the current solver proceeds as follows.

1. **Dominance reduction.** Iteratively eliminate strategies that can be
   removed by the package's dominance checks. In some games this step alone
   solves the game.

2. **Pure-strategy search.** Test the remaining pure profiles for Nash
   equilibrium.

3. **Support enumeration.** Construct candidate subgames/supports for mixed
   and partially mixed equilibria.

4. **Reduce each support.** Apply dominance checks again inside the candidate
   subgame. Supports that cannot contain the desired equilibrium are discarded.

5. **Construct indifference equations.** For every mixing player, expected
   payoffs from actions in the support must agree. One probability per player
   is recovered from the adding-up restriction, so the nonlinear system is
   expressed in the remaining free probabilities.

6. **Solve the nonlinear system.** In the default method, the free probabilities
   initially lie in interval boxes contained in \([0,1]^d\). Interval
   contraction and subdivision are used to search the region systematically.

7. **Newton refinement.** Candidate solution regions are refined to point
   solutions.

8. **Equilibrium verification and consolidation.** Candidate probability
   vectors are checked against equilibrium conditions and nearby numerical
   duplicates are consolidated.

The `fast` option skips the interval-search stage and instead starts Newton
iteration from randomly drawn points in the relevant mixed-strategy simplices.
This is much faster in many problems, but it does not provide the same
systematic search of the strategy space.

The default procedure is therefore best described as a complete-search
procedure **intended to locate all equilibria** of the finite game. As with any
numerical procedure, difficult or degenerate games should be inspected
carefully.


## 7. A short introduction to interval methods

The interval component is what distinguishes the default mixed-strategy search
from simply running Newton's method from a collection of guesses.

Let

\[
[x]=[\underline{x},\overline{x}]
\]

denote an interval known to contain a scalar \(x\). Basic interval operations
are defined so that the resulting interval contains every possible real result.

Addition:

\[
[a,b]+[c,d]=[a+c,b+d].
\]

Subtraction:

\[
[a,b]-[c,d]=[a-d,b-c].
\]

Multiplication:

\[
[a,b][c,d]
=
[\min(ac,ad,bc,bd),\max(ac,ad,bc,bd)].
\]

Floating-point implementations use **outward rounding**: lower bounds are
rounded downward and upper bounds upward so that numerical rounding does not
incorrectly exclude feasible values.


### 7.1 Inclusion functions and pessimism

Replacing ordinary arithmetic in an expression by interval arithmetic produces
an inclusion function \([f]([x])\) satisfying

\[
f([x])\subseteq [f]([x]).
\]

The inclusion can be wider than the true range because the same variable may
appear repeatedly in the expression. This is often called the dependency
problem or interval pessimism.

For example, if \(x\in[-1,2]\), the true range of \(x^2\) is \([0,4]\).
Evaluating the algebraically equivalent expression \(x\times x\) by naive
interval multiplication gives

\[
[-1,2]\times[-1,2]=[-2,4],
\]

which contains the true range but is wider.

This apparent conservatism is useful for exclusion. If zero is **not** contained
in the interval evaluation of an equation over a box, no root can lie anywhere
inside that box.


> **Editorial note.** The earlier slide presentation gave the true range of
> \(x^2\) on \([-1,2]\) as \([1,4]\). That appears to be a typographical error;
> the range is \([0,4]\). The example and its intended point about interval
> pessimism are otherwise unchanged here.


### 7.2 Branch, contract, and exclude

Suppose a scalar root is sought on an initial interval \([x]\).

A generic interval search repeats the following operations:

1. Evaluate an interval inclusion of the equation.
2. If \(0\notin[f]([x])\), discard the interval.
3. If zero remains possible, attempt to **contract** the interval.
4. If the surviving interval is still too wide, bisect it and place the two
   smaller intervals back on the work list.
5. Continue until no unresolved boxes remain.
6. Refine sufficiently small candidate boxes with a point method such as
   Newton iteration.

In several dimensions, intervals become boxes and the Jacobian becomes an
interval matrix. A Taylor-type inclusion has the schematic form

\[
\mathbf f([\mathbf x])
\subseteq
\mathbf f(\mathbf m)
+
[\mathbf J]([\mathbf x])([\mathbf x]-\mathbf m),
\]

where \(\mathbf m\) is typically a point such as the midpoint of the box.

The interval solver used by `dagamesolve` performs equation-by-equation
contraction and subdivision through the companion `intsolver` package.


## 8. Example 1: a two-player coordination game

Consider the game

| | Player 2: 1 | Player 2: 2 |
|---|---:|---:|
| **Player 1: 1** | \(1,2\) | \(0,0.1\) |
| **Player 1: 2** | \(0,0\) | \(3,1\) |

The game has two pure-strategy equilibria and one mixed equilibrium.

The code below constructs the action profiles with `dagamestrats`, enters the
payoffs in matching profile order, and solves the game.


In [ ]:
clear
set obs 2

gen game = 1
gen acts = 2
dagamestrats acts, group(game) generate(s)

* Profile 1: (1,1) -> payoffs (1,2)
gen pay1 = 1
replace pay1 = 2 in 2

* Profile 2: (1,2) -> payoffs (0,.1)
gen pay2 = 0
replace pay2 = .1 in 2

* Profile 3: (2,1) -> payoffs (0,0)
gen pay3 = 0

* Profile 4: (2,2) -> payoffs (3,1)
gen pay4 = 3
replace pay4 = 1 in 2

list game acts s1-s4 pay1-pay4, noobs

dagamesolve s1-s4, group(game) payoffs(pay1-pay4) equilibria(eq)

list eq*, noobs
return list


If `equilibria(eq)` is specified, strategy probabilities are stored in variables
of the form

\[
\texttt{eq\_q\_p\_a},
\]

where `q` indexes the equilibrium, `p` the player, and `a` the action.
The complete strategy profile is stored on the last player observation for the
game. Variables `eq_rets_q` contain the player-specific equilibrium payoffs,
and `eq_count` records the number of equilibria found.


## 9. Example 2: matching pennies

Matching pennies has no pure-strategy equilibrium. Its unique Nash equilibrium
is fully mixed, with each player placing probability \(1/2\) on each action.

This is a useful minimal example because a solver restricted to pure strategies
would report no equilibrium, while a mixed-strategy solver should recover the
interior solution.


In [ ]:
clear
set obs 2

gen game = 1
gen acts = 2
dagamestrats acts, group(game) generate(s)

* Player 1:  1, -1, -1,  1
* Player 2: -1,  1,  1, -1
gen pay1 = cond(_n==1,  1, -1)
gen pay2 = cond(_n==1, -1,  1)
gen pay3 = cond(_n==1, -1,  1)
gen pay4 = cond(_n==1,  1, -1)

dagamesolve s1-s4, group(game) payoffs(pay1-pay4) equilibria(eq)

list eq*, noobs
return list


## 10. Example 3: a three-player coordination game

The earlier presentation used the following three-player game to illustrate
the list representation. In the presentation actions were numbered 0 and 1;
here they are numbered 1 and 2 to match `dagamestrats`.

Only the two unanimous profiles have nonzero payoffs:

\[
\Pi(1,1,1)=(2,1,4),
\qquad
\Pi(2,2,2)=(1.2,2,3),
\]

and the six remaining profiles give all players zero.

The presentation identifies three equilibria in this game. The following cell
constructs and solves it directly.


In [ ]:
clear
set obs 3

gen game = 1
gen acts = 2
dagamestrats acts, group(game) generate(s)

forvalues j = 1/8 {
    gen pay`j' = 0
}

* Unanimous action 1
replace pay1 = 2 in 1
replace pay1 = 1 in 2
replace pay1 = 4 in 3

* Unanimous action 2
replace pay8 = 1.2 in 1
replace pay8 = 2   in 2
replace pay8 = 3   in 3

list s1-s8 pay1-pay8, noobs

dagamesolve s1-s8, group(game) payoffs(pay1-pay8) equilibria(eq)

list eq*, noobs
return list


## 11. The distinctive feature: solving a sequence of games

For empirical work, it is often more useful to solve a family of related games
than a single game.

Consider a two-firm entry game. Each player chooses **Enter** or **Stay Out**.
Normalize the payoff from staying out to zero. If a firm enters alone, its
payoff is \(\pi>0\). If both firms enter, each receives

\[
\pi-\delta.
\]

Thus:

- when \(\delta<\pi\), entry is a dominant action and the unique equilibrium is
  joint entry;
- when \(\delta>\pi\), the game has two asymmetric pure equilibria and a mixed
  equilibrium;
- the knife-edge case \(\delta=\pi\) is deliberately omitted below.

Instead of solving ten games one at a time, we place all ten in one Stata
dataset and call `dagamesolve` once.


In [ ]:
clear
set obs 20

gen game = ceil(_n/2)
bysort game: gen player = _n

gen acts = 2
dagamestrats acts, group(game) generate(s)

* Common monopoly-entry payoff
gen pi = 1

* delta = .5,.6,.7,.8,.9,1.1,1.2,1.3,1.4,1.5
gen delta = .4 + .1*game
replace delta = delta + .1 if delta >= 1

* s1 = (Enter,Enter)
gen pay1 = pi - delta

* s2 = (Enter,StayOut)
gen pay2 = cond(player==1, pi, 0)

* s3 = (StayOut,Enter)
gen pay3 = cond(player==1, 0, pi)

* s4 = (StayOut,StayOut)
gen pay4 = 0

dagamesolve s1-s4, group(game) payoffs(pay1-pay4) equilibria(eq)

list game delta eq_count if player==2, noobs sep(0)


The equilibrium count should change as \(\delta\) crosses \(\pi\). Because the
equilibrium count is stored on the last observation of each game, a simple plot
can summarize the sequence.


In [ ]:
sort delta

twoway connected eq_count delta if player==2, ///
    ytitle("Number of Nash equilibria") ///
    xtitle("Competition parameter delta") ///
    title("Equilibrium multiplicity in a sequence of entry games")


This example captures an important intended use of the software. A researcher
can generate game-specific payoffs from covariates or parameter values, solve
the entire sequence, and then feed equilibrium objects into subsequent
analysis without moving between Stata and a stand-alone game-theory program.


## 12. Complete search, `fast`, and `pureonly`

The default mixed-strategy method uses interval search followed by Newton
refinement. Two useful alternatives are available.

### `fast`

With `fast`, the interval-search stage is skipped. For each support, random
points are drawn from the relevant mixed-strategy simplices and used as starting
values for Newton iteration. `randompoints(#)` controls the number of starts.

This may be dramatically faster, but it can miss equilibria if none of the
random starting points fall in a useful basin of attraction.

### `pureonly`

With `pureonly`, no mixed-strategy systems are solved. This is useful when only
pure equilibria are substantively relevant or when one wants a quick
preliminary diagnostic.


In [ ]:
* Continue from the entry-game dataset above

set seed 5150

dagamesolve s1-s4, group(game) payoffs(pay1-pay4) ///
    equilibria(ef) fast randompoints(200)

dagamesolve s1-s4, group(game) payoffs(pay1-pay4) ///
    equilibria(ep) pureonly

list game delta eq_count ef_count ep_count if player==2, noobs sep(0)


For this simple sequence one would expect the default and sufficiently thorough
`fast` searches to agree. The `pureonly` count differs when a mixed equilibrium
is present. Agreement in a simple example should not be interpreted as a
general completeness guarantee for `fast`.


## 13. Numerical controls

The default complete-search settings are:

| Option | Default | Role |
|---|---:|---|
| `mbwidth(#)` | `1e-4` | Target maximum interval-box width before refinement |
| `tolerance(#)` | `1e-12` | Newton convergence tolerance |
| `coltol(#)` | `1e-4` | Tolerance for consolidating numerically nearby solutions |
| `digits(#)` | `1e-8` | Precision parameter used in interval outward rounding |
| `mnits(#)` | `100` | Maximum Newton iterations |

For the fast search,

| Option | Default | Role |
|---|---:|---|
| `randompoints(#)` | `50` | Random Newton starting points for each candidate support |
| `tolerance(#)` | `1e-12` | Newton convergence tolerance |
| `mnits(#)` | `100` | Maximum Newton iterations |

These parameters control different parts of the numerical procedure and should
not be interpreted as interchangeable measures of "accuracy." In particular,
`mbwidth()` concerns the interval search, `tolerance()` concerns Newton
iteration, `coltol()` concerns consolidation of candidate solutions, and
`digits()` concerns interval arithmetic.


## 14. A sequence-of-games regression test

The package's test suite contains a useful compact example: three two-player
games are stacked in one dataset.

1. a coordination game with three equilibria;
2. a dominant-strategy game with one equilibrium;
3. matching pennies with one mixed equilibrium.

The example below is adapted directly from the current regression test and is
useful both pedagogically and as a reproducibility check.


In [ ]:
clear
set obs 6

gen game = ceil(_n/2)
bysort game: gen player = _n

gen acts = 2
dagamestrats acts, group(game) generate(s)

forvalues j = 1/4 {
    gen pay`j' = .
}

* Game 1: coordination
replace pay1 = 1 if game==1
replace pay2 = 0 if game==1
replace pay3 = 0 if game==1
replace pay4 = 1 if game==1

* Game 2: action 2 strictly dominant for both players
replace pay1 = 3 if game==2 & player==1
replace pay2 = 0 if game==2 & player==1
replace pay3 = 5 if game==2 & player==1
replace pay4 = 1 if game==2 & player==1

replace pay1 = 3 if game==2 & player==2
replace pay2 = 5 if game==2 & player==2
replace pay3 = 0 if game==2 & player==2
replace pay4 = 1 if game==2 & player==2

* Game 3: matching pennies
replace pay1 =  1 if game==3 & player==1
replace pay2 = -1 if game==3 & player==1
replace pay3 = -1 if game==3 & player==1
replace pay4 =  1 if game==3 & player==1

replace pay1 = -1 if game==3 & player==2
replace pay2 =  1 if game==3 & player==2
replace pay3 =  1 if game==3 & player==2
replace pay4 = -1 if game==3 & player==2

dagamesolve s1-s4, group(game) payoffs(pay1-pay4) equilibria(e)

list game player e_count if !missing(e_count), noobs sep(0)

assert e_count==3 if game==1 & player==2
assert e_count==1 if game==2 & player==2
assert e_count==1 if game==3 & player==2

display as result "PASS: sequence of games solved correctly"


## 15. Stored results

`dagamesolve` stores summary information in `r()` in addition to the variables
it creates in the dataset.

Core stored results include:

- `r(playeractions)`: number of actions available to each player;
- `r(totalactions)`: total number of player-actions;
- `r(method)`: `complete` or `newton`;
- `r(games)`: number of games solved;
- `r(meaneqs)`: mean equilibrium count across games;
- `r(maxeqs)`: maximum equilibrium count;
- `r(mineqs)`: minimum equilibrium count;
- `r(sdeqs)`: standard deviation of equilibrium counts.

The complete method also records the interval and Newton settings. The fast
method records the number of random starting points.

Because the equilibrium probability variables remain in the dataset, they can
be reshaped, summarized, merged to market-level data, or used in subsequent
programs without leaving Stata.


In [ ]:
return list


## 16. Computational considerations

The computational burden grows quickly for two related reasons.

First, the number of pure profiles is

\[
\prod_i K_i.
\]

Second, mixed-equilibrium search requires consideration of candidate supports.
For each surviving support, the solver constructs and solves a nonlinear system.
Adding players or actions therefore affects both the size of individual
systems and the number of systems that may have to be considered.

Interval methods are intentionally more demanding than point-based Newton
searches. Their advantage is that boxes can be ruled out over entire regions
of the strategy space rather than only explored from selected starting points.

Several practical implications follow:

- use `pureonly` when mixed equilibria are irrelevant;
- use `fast` when exploratory speed matters more than systematic coverage;
- use the default method when the goal is a systematic search for pure and
  mixed equilibria;
- exploit sequences of games when many games have the same action structure;
- inspect difficult, degenerate, or nearly duplicate equilibria carefully;
- report numerical settings in applications where equilibrium multiplicity is
  substantively important.


## 17. What this makes possible for empirical work

The original motivation for these routines was closely connected to empirical
models of strategic interaction.

Suppose market \(m\) is associated with a game whose payoff primitives depend
on observed covariates \(X_m\) and parameters \(\theta\):

\[
\Pi_m=\Pi(X_m,\theta).
\]

For a trial value of \(\theta\), one can:

1. construct the payoff matrix for every market;
2. solve the sequence of games;
3. obtain the equilibrium correspondence market by market;
4. compare predicted equilibrium behavior with observed outcomes;
5. embed the calculation in an estimation, simulation, or counterfactual
   procedure.

The computational problem is therefore not incidental. When a structural model
permits multiple equilibria, the ability to obtain the equilibrium
correspondence repeatedly can be an essential piece of the empirical method.

Bajari, Hong, and Ryan (2010) provide an important example of the broader
connection between discrete games and econometric analysis.


## 18. Conclusions

`dagamesolve` is designed to make finite-game equilibrium computation available
inside a data-analysis environment.

The easy part of a finite game is often the part one sees in a textbook:
enumerate a few pure profiles and check unilateral deviations. The difficult
part is what happens once mixed and partially mixed equilibria are admitted.
The problem becomes one of support enumeration and nonlinear equation solving.

The package addresses that problem by combining game-specific reductions with
general numerical tools:

- dominance reduction;
- direct pure-strategy checks;
- support enumeration;
- interval contraction and subdivision;
- Newton refinement;
- candidate verification and consolidation.

The ability to apply the same machinery to a sequence of games is particularly
useful for applications in which payoffs vary across markets, observations, or
counterfactual parameter values.

The present notebook is intended to serve simultaneously as documentation,
a reproducible collection of examples, and a technical description of the
solver.


## References

Bajari, Patrick, Han Hong, and Stephen P. Ryan. 2010. “Identification and
Estimation of a Discrete Game of Complete Information.” *Econometrica*
78(5): 1529–1568.

Jaulin, Luc, Michel Kieffer, Olivier Didrit, and Eric Walter. 2001.
*Applied Interval Analysis*. London: Springer.

Wilson, Robert. 1971. “Computing Equilibria of N-Person Games.”
*SIAM Journal on Applied Mathematics* 21(1): 80–87.

**To add in the next draft:** the full Kearfott (1990) reference used in the
earlier interval-contraction presentation, and any additional game-computation
or econometric references desired for a journal version.


---

### Development notes for the next draft

- Decide whether the title should emphasize **all equilibria**, **sequences of
  games**, or both.
- Add actual executed Stata output after the notebook is opened with the Stata
  kernel.
- Consider a timing table comparing the default and `fast` methods.
- Consider an example with heterogeneous numbers of actions across players.
- Add a short formal description of support enumeration if the document is
  aimed at a journal rather than primarily at users.
- Add the complete Kearfott reference and any additional literature on
  equilibrium computation.
- Once the prose stabilizes, export the executed notebook to PDF and add the
  notebook and PDF to the `DaGameSolve` repository.
